In [ ]:
# =============================================================================
# OFFSIDE 2026 — FOOTBALL DATATHON
# FILE 1: preprocess.py — Cleaned & Insulated
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = "/kaggle/input/competitions/offside-data-thon"
OUT_PATH  = "/kaggle/working" 

# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("Loading raw data...")
train_raw = pd.read_csv(f"{DATA_PATH}/train.csv")
test_raw  = pd.read_csv(f"{DATA_PATH}/test.csv")

print(f"Train shape: {train_raw.shape} | Test shape: {test_raw.shape}")
print(f"Positive target rate: {train_raw['scored_flag'].mean():.4f}")

TARGET      = 'scored_flag'
global_rate = train_raw[TARGET].mean()

# =============================================================================
# 2. DROP COLS
# =============================================================================
DROP_COLS = [
    'goal_diff_abs',
    'appearance_id',
    'date',
    'name_x',
    'name_y',
    'home_club_name',
    'away_club_name',
    'stadium',
    'referee',
    'home_club_id',
    'away_club_id',
]

# =============================================================================
# 3. GOAL CONTEXT FEATURES
# =============================================================================
def add_goal_context(df):
    df = df.copy()
    is_home = df['home_away'].str.upper().str.strip() == 'HOME'

    df['my_team_goals']     = np.where(is_home, df['home_club_goals'], df['away_club_goals'])
    df['opponent_goals']    = np.where(is_home, df['away_club_goals'], df['home_club_goals'])
    df['total_match_goals'] = df['home_club_goals'] + df['away_club_goals']

    df['team_won']  = (df['my_team_goals'] > df['opponent_goals']).astype(int)
    df['team_drew'] = (df['my_team_goals'] == df['opponent_goals']).astype(int)
    df['team_lost'] = (df['my_team_goals'] < df['opponent_goals']).astype(int)

    df['high_scoring']      = (df['total_match_goals'] >= 4).astype(int)
    df['team_dominant']     = (df['my_team_goals'] >= 3).astype(int)
    df['team_scored_2plus'] = (df['my_team_goals'] >= 2).astype(int)
    df['team_scored_3plus'] = (df['my_team_goals'] >= 3).astype(int)

    return df

train_raw = add_goal_context(train_raw)
test_raw  = add_goal_context(test_raw)
print("✅ Goal context features processed.")

# =============================================================================
# 4. CORE FEATURE ENGINEERING
# =============================================================================
def engineer_features(df):
    df = df.copy()

    df['has_xg_data'] = df['avg_xG'].notna().astype(int)

    xg_cols = ['avg_xG','avg_npxG','avg_shots','avg_xA',
               'avg_key_passes','avg_xGChain','avg_xGBuildup','xG_to_xA_ratio']
    for col in xg_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df['goal_threat_score'] = (
        df['avg_xG']                    * 0.4 +
        df['avg_npxG']                  * 0.3 +
        df['avg_shots']                 * 0.01 +
        df['finisher_flag'].astype(int) * 0.5 +
        df['is_attacker'].astype(int)   * 0.3
    )

    df['minutes_quality']  = df['minutes_played'] * df['starter_flag'].astype(int)
    df['mins_weighted_xG'] = df['avg_xG'] * (df['minutes_played'] / 90)
    df['mins_per_xG']      = df['minutes_played'] / (df['avg_xG'] + 0.01)

    df['intl_pedigree'] = (
        df['international_goals'].fillna(0) * 2 +
        df['international_caps'].fillna(0)  * 0.1
    )

    df['market_value_before_match'] = df['market_value_before_match'].fillna(
        df['market_value_before_match'].median()
    )
    df['log_market_value'] = df['log_market_value'].fillna(0)

    pos_map = {
        'Centre-Forward': 4.0, 'Second Striker': 4.0,
        'Left Winger': 3.0,    'Right Winger': 3.0,    'Attacking Midfield': 3.0,
        'Central Midfield': 2.0,'Left Midfield': 2.0,  'Right Midfield': 2.0,
        'Defensive Midfield': 1.0,
        'Centre-Back': 0.5,    'Left-Back': 0.5,       'Right-Back': 0.5,
        'Goalkeeper': 0.0
    }
    df['position_attack_score'] = df['sub_position'].map(pos_map).fillna(1.0)

    df['elite_attacker'] = (
        (df['is_attacker'].astype(int) == 1) &
        (df['avg_xG'] > df['avg_xG'].quantile(0.75))
    ).astype(int)

    df['impact_sub'] = (
        (df['substitute_flag'].astype(int) == 1) &
        (df['minutes_played'] > 30)
    ).astype(int)

    df['full_match_attacker'] = (
        (df['is_attacker'].astype(int) == 1) &
        (df['full_match_flag'].astype(int) == 1)
    ).astype(int)

    mv_q75 = df['market_value_before_match'].quantile(0.75)
    df['highvalue_attacker'] = (
        (df['is_attacker'].astype(int) == 1) &
        (df['market_value_before_match'] > mv_q75)
    ).astype(int)

    df['is_home'] = (df['home_away'].str.upper().str.strip() == 'HOME').astype(int)

    df['goal_per_cap']         = df['goal_per_cap'].fillna(0)
    df['international_caps']   = df['international_caps'].fillna(0)
    df['international_goals']  = df['international_goals'].fillna(0)
    df['height_in_cm']         = df['height_in_cm'].fillna(df['height_in_cm'].median())
    df['age']                  = df['age'].fillna(df['age'].median())
    df['attendance']           = df['attendance'].fillna(df['attendance'].median())

    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

    return df

train_raw = engineer_features(train_raw)
test_raw  = engineer_features(test_raw)
print("✅ Core feature engineering done.")

# =============================================================================
# 5. HIGH-CARDINALITY TARGET ENCODING (PLAYER ONLY)
# Low-cardinality categories dropped from LOO loop to protect floating points
# =============================================================================
def loo_encode_player(train_df, test_df, group_col, target_col, new_col, k, global_r):
    stats = (train_df.groupby(group_col)[target_col]
                     .agg(['sum','count'])
                     .reset_index()
                     .rename(columns={'sum':'_tot_g','count':'_tot_a'}))
    
    train_df = train_df.merge(stats, on=group_col, how='left')
    
    train_df['_loo_g'] = train_df['_tot_g'] - train_df[target_col]
    train_df['_loo_a'] = (train_df['_tot_a'] - 1).clip(lower=0)
    train_df[new_col]  = (train_df['_loo_g'] + k * global_r) / (train_df['_loo_a'] + k)
    train_df[new_col].fillna(global_r, inplace=True)
    train_df.drop(columns=['_tot_g','_tot_a','_loo_g','_loo_a'], inplace=True)

    stats[new_col] = (stats['_tot_g'] + k * global_r) / (stats['_tot_a'] + k)
    test_df = test_df.merge(stats[[group_col, new_col]], on=group_col, how='left')
    test_df[new_col].fillna(global_r, inplace=True)

    return train_df, test_df

train_raw['_name_y'] = pd.read_csv(f"{DATA_PATH}/train.csv")['name_y'].values
test_raw['_name_y']  = pd.read_csv(f"{DATA_PATH}/test.csv")['name_y'].values

train_raw, test_raw = loo_encode_player(train_raw, test_raw, '_name_y', TARGET, 'player_score_rate_smooth', 50, global_rate)

# Clean out temporary identity reference string tags
train_raw.drop(columns=['_name_y'], errors='ignore', inplace=True)
test_raw.drop(columns=['_name_y'], errors='ignore', inplace=True)

print("✅ Target encoding completed securely.")

# =============================================================================
# 6. INTERACTION FEATURES
# =============================================================================
def add_interaction_features(df):
    df = df.copy()

    df['player_rate_x_minutes']  = df['player_score_rate_smooth'] * df['minutes_played']
    df['player_rate_x_xG']       = df['player_score_rate_smooth'] * df['avg_xG']

    df['my_team_goals_x_xG']        = df['my_team_goals'] * df['avg_xG']
    df['my_team_goals_x_rate']       = df['my_team_goals'] * df['player_score_rate_smooth']
    df['my_team_goals_x_position']   = df['my_team_goals'] * df['position_attack_score']
    df['my_team_goals_x_minutes']    = df['my_team_goals'] * df['minutes_played']
    df['team_goals_x_elite']         = df['my_team_goals'] * df['elite_attacker']
    df['team_goals_x_mins_weighted'] = df['my_team_goals'] * df['avg_xG'] * (df['minutes_played'] / 90)
    df['goal_opportunity']           = df['my_team_goals'] * df['position_attack_score']
    df['xG_x_team_goals']            = df['avg_xG'] * df['my_team_goals']

    df['dominant_team_attacker'] = ((df['my_team_goals'] >= 3) & (df['is_attacker'].astype(int) == 1)).astype(int)
    df['highscoring_attacker']   = ((df['total_match_goals'] >= 4) & (df['is_attacker'].astype(int) == 1)).astype(int)

    p90 = df['player_score_rate_smooth'].quantile(0.90)
    df['elite_striker'] = (
        (df['player_score_rate_smooth'] >= p90) &
        (df['is_attacker'].astype(int) == 1) &
        (df['minutes_played'] >= 60)
    ).astype(int)

    df['xG_x_mins_weighted'] = df['avg_xG'] * (df['minutes_played'] / 90) * df['player_score_rate_smooth']

    return df

train_raw = add_interaction_features(train_raw)
test_raw  = add_interaction_features(test_raw)
print("✅ Interaction architecture compiled.")

# =============================================================================
# 7. CATEGORICAL ENCODING
# =============================================================================
CAT_COLS = [
    'position','sub_position','foot','home_away',
    'competition_type','confederation','age_bucket',
    'market_value_tier','country_name','country_of_citizenship'
]

le = LabelEncoder()
for col in CAT_COLS:
    if col in train_raw.columns and col in test_raw.columns:
        combined = pd.concat([train_raw[col], test_raw[col]], axis=0).astype(str).fillna('Unknown')
        le.fit(combined)
        train_raw[col] = le.transform(train_raw[col].astype(str).fillna('Unknown'))
        test_raw[col]  = le.transform(test_raw[col].astype(str).fillna('Unknown'))

print("✅ Categorical mapping complete.")

# =============================================================================
# 8. FINAL FEATURE MATRIX SPECIFICATION
# =============================================================================
EXCLUDE = DROP_COLS + [TARGET, 'min_bucket',
                       'home_club_name','away_club_name',
                       'stadium','referee','name_x','name_y',
                       'home_club_id','away_club_id']

FEATURES = [
    c for c in train_raw.columns
    if c not in EXCLUDE
    and train_raw[c].dtype != 'object'
]

X      = train_raw[FEATURES].copy()
y      = train_raw[TARGET].astype(int).copy()
X_test = test_raw[FEATURES].copy()

# Median fallback cleanup
for col in FEATURES:
    if X[col].isnull().sum() > 0:
        fill_val = X[col].median() if X[col].dtype in ['float64','int64','float32'] else 0
        X[col]      = X[col].fillna(fill_val)
        X_test[col] = X_test[col].fillna(fill_val)

# =============================================================================
# 9. VERIFICATION LOG
# =============================================================================
print("\n" + "="*50)
print("PREPROCESS PRODUCTION VERIFICATION")
print("="*50)
print(f"X shape:         {X.shape}")
print(f"X_test shape:    {X_test.shape}")
print(f"Total features:  {len(FEATURES)}")

corr = X.corrwith(y.astype(float)).abs().sort_values(ascending=False)
print(f"Max point correlation detected: {corr.max():.4f}")

assert X.isnull().sum().sum() == 0, "Null pointer detected inside X!"
assert X.shape[1] == X_test.shape[1], "Feature layout count mismatch!"

# =============================================================================
# 10. SAVE
# =============================================================================
test_ids = pd.read_csv(f"{DATA_PATH}/test.csv")['appearance_id']

with open(f"{OUT_PATH}/X_train.pkl",  'wb') as f: pickle.dump(X,        f)
with open(f"{OUT_PATH}/X_test.pkl",   'wb') as f: pickle.dump(X_test,   f)
with open(f"{OUT_PATH}/y_train.pkl",  'wb') as f: pickle.dump(y,        f)
with open(f"{OUT_PATH}/features.pkl", 'wb') as f: pickle.dump(FEATURES, f)
with open(f"{OUT_PATH}/test_ids.pkl", 'wb') as f: pickle.dump(test_ids, f)

print("\n✅ Cleaned binary arrays saved safely. Run train.py next.")

In [ ]:
# =============================================================================
# OFFSIDE 2026 — FOOTBALL DATATHON
# FILE 2: train.py — Optimized LightGBM Execution Engine
# =============================================================================

import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import lightgbm as lgb

OUT_PATH = "/kaggle/working"

# 1. LOAD PREPROCESSED ARRAYS
print("Loading uncompromised dataset arrays...")
with open(f'{OUT_PATH}/X_train.pkl',  'rb') as f: X        = pickle.load(f)
with open(f'{OUT_PATH}/X_test.pkl',   'rb') as f: X_test   = pickle.load(f)
with open(f'{OUT_PATH}/y_train.pkl',  'rb') as f: y        = pickle.load(f)
with open(f'{OUT_PATH}/test_ids.pkl', 'rb') as f: test_ids = pickle.load(f)

scale_pos = (y == 0).sum() / (y == 1).sum()

# 2. CROSS-VALIDATION STRUCTURAL SCHEDULING
N_SPLITS = 3
SKF = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
ap_folds   = []

lgb_params = {
    'objective':         'binary',
    'metric':            'average_precision',
    'n_estimators':      4000,
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         6,
    'min_child_samples': 2000,
    'feature_fraction':  0.6,
    'bagging_fraction':  0.7,
    'bagging_freq':      5,
    'lambda_l1':         1.0,
    'lambda_l2':         2.0,
    'scale_pos_weight':  scale_pos,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

# 3. PRODUCTION EXECUTION LOOP
print("\n" + "="*55)
print(f"Launching LightGBM Machine via {N_SPLITS}-Fold Stratified Loop")
print("="*55)

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(80, verbose=False),
                   lgb.log_evaluation(100)]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds        += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    ap = average_precision_score(y_val, oof_preds[val_idx])
    ap_folds.append(ap)
    print(f"➡️ Fold {fold} Local Validation AP: {ap:.4f} | Best Iteration: {model.best_iteration_}")

# 4. EXPORT SOLUTION GENERATION
cv_ap = average_precision_score(y, oof_preds)
print(f"\n✅ Total Out-of-Fold Blend CV AP: {cv_ap:.4f}")

sub = pd.DataFrame({
    'appearance_id': test_ids.values,
    'scored_flag':   np.clip(test_preds, 0, 1)
})

sub.to_csv(f'{OUT_PATH}/solution.csv', index=False)
print(f"solution.csv written successfully to workspace disk. Size footprint: {sub.shape}")
print(sub.head())